In [1]:
# 批量点云指标：CD / F1@τ / Completeness / Accuracy / RMSE + ICP Rotation/Translation
import sys
import subprocess
from pathlib import Path
import numpy as np
from scipy.spatial import cKDTree
import trimesh

# ===== 输入目录（每个目录 9 个 glb） =====
GT_DIR = "/home/viplab/liukejia/project/result2/reconstruct_gt_glb/"
PRED_DIR = "/home/viplab/liukejia/project/result2/reconstruct_pred_glb/"

In [2]:

# 采样与阈值设置
MAX_POINTS = 120000
F1_THRESHOLDS = [0.2, 0.5]
ICP_MAX_ITER = 50
ICP_TOL = 1e-6
ICP_SAMPLE_SIZE = 8000

def load_points_from_glb(glb_path: str, max_points: int = 120000, seed: int = 0):
    obj = trimesh.load(glb_path, force="scene")

    if isinstance(obj, trimesh.Trimesh):
        pts = np.asarray(obj.vertices, dtype=np.float64)
    else:
        # Scene -> 合并成单个 mesh（含变换）
        merged = obj.dump(concatenate=True)
        if isinstance(merged, (list, tuple)):
            merged = merged[0] if len(merged) > 0 else None
        if merged is None or not hasattr(merged, "vertices"):
            raise ValueError(f"无法从场景解析顶点: {glb_path}")
        pts = np.asarray(merged.vertices, dtype=np.float64)

    if pts.shape[0] == 0:
        raise ValueError(f"空点云: {glb_path}")

    if max_points is not None and pts.shape[0] > max_points:
        rng = np.random.default_rng(seed)
        idx = rng.choice(pts.shape[0], size=max_points, replace=False)
        pts = pts[idx]
    return pts

def nearest_distances(src_pts: np.ndarray, dst_pts: np.ndarray):
    tree = cKDTree(dst_pts)
    dists, _ = tree.query(src_pts, k=1, workers=-1)
    return dists.astype(np.float64)

def chamfer_distance_l2_squared(p_rec: np.ndarray, p_gt: np.ndarray):
    d_rec = nearest_distances(p_rec, p_gt)
    d_gt = nearest_distances(p_gt, p_rec)
    cd = np.mean(d_rec ** 2) + np.mean(d_gt ** 2)
    return float(cd), d_rec, d_gt

def f1_at_threshold(d_rec: np.ndarray, d_gt: np.ndarray, tau: float):
    precision = float(np.mean(d_rec < tau))
    recall = float(np.mean(d_gt < tau))
    f1 = 0.0 if (precision + recall) == 0 else 2.0 * precision * recall / (precision + recall)
    return precision, recall, f1

def pointcloud_metrics(p_rec: np.ndarray, p_gt: np.ndarray, thresholds=(0.2, 0.5)):
    cd, d_rec, d_gt = chamfer_distance_l2_squared(p_rec, p_gt)
    accuracy = float(np.mean(d_rec))
    completeness = float(np.mean(d_gt))
    rmse_nn = float(np.sqrt((np.mean(d_rec ** 2) + np.mean(d_gt ** 2)) / 2.0))

    out = {
        "cd_l2_squared": cd,
        "accuracy": accuracy,
        "completeness": completeness,
        "rmse_nn": rmse_nn,
        "num_rec": int(p_rec.shape[0]),
        "num_gt": int(p_gt.shape[0]),
        "f1": {},
    }
    for tau in thresholds:
        p, r, f1 = f1_at_threshold(d_rec, d_gt, tau=float(tau))
        out["f1"][float(tau)] = {"precision": p, "recall": r, "f1": f1}
    return out

def best_fit_transform(A: np.ndarray, B: np.ndarray):
    """求解 A->B 的刚体变换（Kabsch）"""
    centroid_A = np.mean(A, axis=0)
    centroid_B = np.mean(B, axis=0)

    AA = A - centroid_A
    BB = B - centroid_B

    H = AA.T @ BB
    U, _, Vt = np.linalg.svd(H)
    R = Vt.T @ U.T

    # 处理反射
    if np.linalg.det(R) < 0:
        Vt[-1, :] *= -1
        R = Vt.T @ U.T

    t = centroid_B - R @ centroid_A
    return R, t

def icp_point_to_point(src: np.ndarray, dst: np.ndarray, max_iter=50, tol=1e-6, sample_size=8000, seed=0):
    """返回将 src 对齐到 dst 的刚体变换 R,t（src_aligned = R@src + t）"""
    rng = np.random.default_rng(seed)
    tree = cKDTree(dst)

    R_total = np.eye(3, dtype=np.float64)
    t_total = np.zeros(3, dtype=np.float64)

    prev_error = np.inf
    src0 = src.astype(np.float64, copy=False)

    for _ in range(max_iter):
        src_trans = (R_total @ src0.T).T + t_total

        if src_trans.shape[0] > sample_size:
            idx = rng.choice(src_trans.shape[0], size=sample_size, replace=False)
            src_trans_s = src_trans[idx]
        else:
            src_trans_s = src_trans

        dists, nn_idx = tree.query(src_trans_s, k=1, workers=-1)
        dst_match = dst[nn_idx]

        dR, dt = best_fit_transform(src_trans_s, dst_match)

        # 累积变换
        R_total = dR @ R_total
        t_total = dR @ t_total + dt

        mean_error = float(np.mean(dists))
        if abs(prev_error - mean_error) < tol:
            break
        prev_error = mean_error

    return R_total, t_total

def rotation_angle_deg_from_R(R: np.ndarray):
    tr = np.trace(R)
    cos_theta = np.clip((tr - 1.0) / 2.0, -1.0, 1.0)
    theta = np.arccos(cos_theta)
    return float(np.degrees(theta))

def evaluate_one_pair(pred_glb: str, gt_glb: str, seed_base: int):
    p_pred = load_points_from_glb(pred_glb, max_points=MAX_POINTS, seed=seed_base)
    p_gt = load_points_from_glb(gt_glb, max_points=MAX_POINTS, seed=seed_base + 1)

    # 原始指标
    m_raw = pointcloud_metrics(p_pred, p_gt, thresholds=F1_THRESHOLDS)

    # ICP 刚体对齐
    R, t = icp_point_to_point(
        p_pred, p_gt,
        max_iter=ICP_MAX_ITER,
        tol=ICP_TOL,
        sample_size=ICP_SAMPLE_SIZE,
        seed=seed_base + 2,
    )
    rot_err_deg = rotation_angle_deg_from_R(R)
    trans_err = float(np.linalg.norm(t))

    # 对齐后指标（可选，通常更公平）
    p_pred_aligned = (R @ p_pred.T).T + t
    m_icp = pointcloud_metrics(p_pred_aligned, p_gt, thresholds=F1_THRESHOLDS)

    return {
        "raw": m_raw,
        "icp": m_icp,
        "icp_rotation_error_deg": rot_err_deg,
        "icp_translation_error": trans_err,
    }

# ===== 批量匹配并评估 =====
gt_files = sorted(Path(GT_DIR).glob("*.glb"))
pred_files = sorted(Path(PRED_DIR).glob("*.glb"))

if len(gt_files) == 0 or len(pred_files) == 0:
    raise FileNotFoundError("GT_DIR 或 PRED_DIR 下没有找到 .glb 文件")

gt_map = {p.name: str(p) for p in gt_files}
pred_map = {p.name: str(p) for p in pred_files}
common_names = sorted(set(gt_map.keys()) & set(pred_map.keys()))

if len(common_names) == 0:
    raise ValueError("两个目录没有同名 glb，无法自动配对")

missing_in_pred = sorted(set(gt_map.keys()) - set(pred_map.keys()))
missing_in_gt = sorted(set(pred_map.keys()) - set(gt_map.keys()))

print(f"GT 文件数: {len(gt_files)}, PRED 文件数: {len(pred_files)}, 成功配对: {len(common_names)}")
if missing_in_pred:
    print("仅 GT 有: ", missing_in_pred)
if missing_in_gt:
    print("仅 PRED 有: ", missing_in_gt)

rows = []
for i, name in enumerate(common_names):
    result = evaluate_one_pair(pred_map[name], gt_map[name], seed_base=1000 + 10 * i)
    rows.append((name, result))

# ===== 打印逐文件结果 =====
for name, r in rows:
    raw = r["raw"]
    icp = r["icp"]
    print("=" * 80)
    print(f"[{name}]")
    print(f"RAW   | CD: {raw['cd_l2_squared']:.6f}, Acc: {raw['accuracy']:.6f}, Comp: {raw['completeness']:.6f}, RMSE: {raw['rmse_nn']:.6f}")
    for tau in F1_THRESHOLDS:
        s = raw["f1"][float(tau)]
        print(f"RAW   | F1@{tau}: P={s['precision']:.4f}, R={s['recall']:.4f}, F1={s['f1']:.4f}")

    print(f"ICP   | RotErr(deg): {r['icp_rotation_error_deg']:.6f}, TransErr: {r['icp_translation_error']:.6f}")
    print(f"ICP   | CD: {icp['cd_l2_squared']:.6f}, Acc: {icp['accuracy']:.6f}, Comp: {icp['completeness']:.6f}, RMSE: {icp['rmse_nn']:.6f}")
    for tau in F1_THRESHOLDS:
        s = icp["f1"][float(tau)]
        print(f"ICP   | F1@{tau}: P={s['precision']:.4f}, R={s['recall']:.4f}, F1={s['f1']:.4f}")

# ===== 汇总平均 =====
def avg(values):
    return float(np.mean(values)) if len(values) else float("nan")

avg_raw_cd = avg([r["raw"]["cd_l2_squared"] for _, r in rows])
avg_raw_acc = avg([r["raw"]["accuracy"] for _, r in rows])
avg_raw_comp = avg([r["raw"]["completeness"] for _, r in rows])
avg_raw_rmse = avg([r["raw"]["rmse_nn"] for _, r in rows])

avg_icp_rot = avg([r["icp_rotation_error_deg"] for _, r in rows])
avg_icp_trans = avg([r["icp_translation_error"] for _, r in rows])
avg_icp_cd = avg([r["icp"]["cd_l2_squared"] for _, r in rows])
avg_icp_acc = avg([r["icp"]["accuracy"] for _, r in rows])
avg_icp_comp = avg([r["icp"]["completeness"] for _, r in rows])
avg_icp_rmse = avg([r["icp"]["rmse_nn"] for _, r in rows])

print("\n" + "#" * 80)
print("平均结果 (ALL PAIRED FILES)")
print(f"RAW   | CD: {avg_raw_cd:.6f}, Acc: {avg_raw_acc:.6f}, Comp: {avg_raw_comp:.6f}, RMSE: {avg_raw_rmse:.6f}")

for tau in F1_THRESHOLDS:
    vals = [r["raw"]["f1"][float(tau)]["f1"] for _, r in rows]
    print(f"RAW   | Mean F1@{tau}: {avg(vals):.4f}")

print(f"ICP   | Mean RotErr(deg): {avg_icp_rot:.6f}, Mean TransErr: {avg_icp_trans:.6f}")
print(f"ICP   | CD: {avg_icp_cd:.6f}, Acc: {avg_icp_acc:.6f}, Comp: {avg_icp_comp:.6f}, RMSE: {avg_icp_rmse:.6f}")
for tau in F1_THRESHOLDS:
    vals = [r["icp"]["f1"][float(tau)]["f1"] for _, r in rows]
    print(f"ICP   | Mean F1@{tau}: {avg(vals):.4f}")

GT 文件数: 105, PRED 文件数: 114, 成功配对: 105
仅 PRED 有:  ['num123_-800_100_0_0007.glb', 'num123_-800_150_0_0008.glb', 'num123_-800_200_0_0009.glb', 'num123_-850_100_0_0004.glb', 'num123_-850_150_0_0005.glb', 'num123_-850_200_0_0006.glb', 'num123_-900_100_0_0001.glb', 'num123_-900_150_0_0002.glb', 'num123_-900_200_0_0003.glb']


/tmp/ipykernel_2650362/19175499.py:15: DeprecationWarning: `Scene.dump(concatenate=True)` DEPRECATED FOR REMOVAL APRIL 2025: replace with `Scene.to_geometry()`
  merged = obj.dump(concatenate=True)


[num123_-1000_0_0_0064.glb]
RAW   | CD: 0.023007, Acc: 0.099661, Comp: 0.099661, RMSE: 0.107255
RAW   | F1@0.2: P=1.0000, R=1.0000, F1=1.0000
RAW   | F1@0.5: P=1.0000, R=1.0000, F1=1.0000
ICP   | RotErr(deg): 0.000002, TransErr: 0.077513
ICP   | CD: 0.010999, Acc: 0.071875, Comp: 0.071875, RMSE: 0.074159
ICP   | F1@0.2: P=1.0000, R=1.0000, F1=1.0000
ICP   | F1@0.5: P=1.0000, R=1.0000, F1=1.0000
[num123_-1000_100_0_0066.glb]
RAW   | CD: 1.962086, Acc: 0.911872, Comp: 0.927112, RMSE: 0.990476
RAW   | F1@0.2: P=0.1429, R=0.1429, F1=0.1429
RAW   | F1@0.5: P=0.1429, R=0.1429, F1=0.1429
ICP   | RotErr(deg): 0.022027, TransErr: 0.974261
ICP   | CD: 0.719422, Acc: 0.534036, Comp: 0.576673, RMSE: 0.599759
ICP   | F1@0.2: P=0.0000, R=0.0000, F1=0.0000
ICP   | F1@0.5: P=0.6429, R=0.5714, F1=0.6050
[num123_-1000_150_0_0067.glb]
RAW   | CD: 0.669844, Acc: 0.537748, Comp: 0.537748, RMSE: 0.578725
RAW   | F1@0.2: P=0.1429, R=0.1429, F1=0.1429
RAW   | F1@0.5: P=0.1429, R=0.1429, F1=0.1429
ICP   | RotE

In [ ]:
import os
import re
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from pathlib import Path
import trimesh
from scipy.spatial import KDTree
from PIL import Image

# ========== 配置 ==========
PRED_DIR = Path("/home/viplab/liukejia/project/result2/reconstruct_pred_glb")
GT_DIR   = Path("/home/viplab/liukejia/project/result2/reconstruct_gt_glb")
OUT_DIR  = Path("/home/viplab/liukejia/project/result2/figures")
OUT_DIR.mkdir(exist_ok=True, parents=True)

IMG_SIZE = (512, 512)
CAM_ELEV = 30
CAM_AZIM = -45
CAM_DIST = 0.3
CAM_ZOOM = 0.4
# 参数：稀疏点云使用手动投影（大方块），密集点云使用 matplotlib scatter（圆形抗锯齿）
SPARSE_THRESHOLD = 25000       # 点数低于此值视为稀疏
POINT_SIZE_SPARSE = 5          # 手动投影时的方块半宽
POINT_SIZE_DENSE = 2.5          # scatter 的点大小

# CD 计算与选取参数
CD_DOWNSAMPLE = 5000           # 计算 CD 时的采样点数
NUM_OVERVIEW = 16              # 图6：概览 tile 数
NUM_CHALLENGE = 6              # 图7：挑战案例数

def parse_coords(filename):
    parts = Path(filename).stem.split('_')
    if len(parts) >= 3:
        try:
            return int(parts[1]), int(parts[2])
        except:
            pass
    return None

def load_points_and_colors(glb_path):
    """返回 (points, colors)，colors 为 (N,3) uint8 或 None"""
    obj = trimesh.load(glb_path)
    points_list = []
    colors_list = []
    def collect(item):
        if isinstance(item, trimesh.Scene):
            for geom in item.geometry.values():
                collect(geom)
        elif isinstance(item, trimesh.PointCloud):
            if len(item.vertices) > 0:
                points_list.append(item.vertices)
                if item.colors is not None:
                    colors = item.colors
                    if colors.shape[1] == 4:
                        colors = colors[:, :3]
                    if colors.max() <= 1.0:
                        colors = (colors * 255).astype(np.uint8)
                    colors_list.append(colors)
                else:
                    colors_list.append(None)
        elif isinstance(item, trimesh.Trimesh):
            if len(item.vertices) > 0:
                points_list.append(item.vertices)
                if hasattr(item.visual, 'vertex_colors') and item.visual.vertex_colors is not None:
                    vc = item.visual.vertex_colors
                    if vc.shape[1] == 4:
                        vc = vc[:, :3]
                    if vc.max() <= 1.0:
                        vc = (vc * 255).astype(np.uint8)
                    colors_list.append(vc)
                else:
                    colors_list.append(None)
    collect(obj)
    if not points_list:
        return np.empty((0,3)), None
    all_pts = np.vstack(points_list)
    all_colors = None
    if all(c is not None for c in colors_list):
        all_colors = np.vstack(colors_list)
        if len(all_colors) != len(all_pts):
            all_colors = None
    return all_pts, all_colors

def normalize_points(points):
    if len(points) == 0:
        return points
    center = points.mean(axis=0)
    points_centered = points - center
    max_radius = np.linalg.norm(points_centered, axis=1).max()
    if max_radius > 0:
        points_centered /= max_radius
    return points_centered

def project_points(points, elev, azim, dist, img_size):
    """投影到像素坐标，返回 (u, v, depth) 有效点"""
    if len(points) == 0:
        return np.array([]), np.array([]), np.array([])
    elev_rad = np.radians(elev)
    azim_rad = np.radians(azim)
    cx = dist * np.cos(elev_rad) * np.cos(azim_rad)
    cy = dist * np.cos(elev_rad) * np.sin(azim_rad)
    cz = dist * np.sin(elev_rad)
    forward = -np.array([cx, cy, cz])
    forward = forward / np.linalg.norm(forward)
    world_up = np.array([0, 1, 0])
    right = np.cross(world_up, forward)
    right = right / np.linalg.norm(right)
    up = np.cross(forward, right)
    R = np.vstack([right, up, forward])
    points_cam = points - np.array([cx, cy, cz])
    points_cam = points_cam @ R.T
    fov_rad = np.radians(45)
    focal = 1.0 / np.tan(fov_rad/2)
    x_proj = points_cam[:,0] * focal / points_cam[:,2]
    y_proj = points_cam[:,1] * focal / points_cam[:,2]
    width, height = img_size
    u = (x_proj + 1.0) * 0.5 * width
    v = (1.0 - (y_proj + 1.0) * 0.5) * height
    valid = (points_cam[:,2] > 0.01) & (u >= 0) & (u < width) & (v >= 0) & (v < height)
    return u[valid], v[valid], points_cam[valid, 2]

def render_manual(points, colors, img_size, elev, azim, dist, point_size):
    """手动绘制方块，适用于稀疏点云（带纹理颜色或深度着色）"""
    u, v, depth = project_points(points, elev, azim, dist, img_size)
    if len(u) == 0:
        return np.ones((img_size[1], img_size[0], 3), dtype=np.uint8) * 240
    order = np.argsort(depth)[::-1]
    u = u[order].astype(int)
    v = v[order].astype(int)
    img = np.ones((img_size[1], img_size[0], 3), dtype=np.uint8) * 255
    if colors is not None and len(colors) == len(points):
        colors_sorted = colors[order]
        if colors_sorted.dtype != np.uint8:
            if colors_sorted.max() <= 1.0:
                colors_sorted = (colors_sorted * 255).astype(np.uint8)
            else:
                colors_sorted = colors_sorted.astype(np.uint8)
        color_img = colors_sorted
    else:
        norm_depth = (depth[order] - depth.min()) / (depth.max() - depth.min() + 1e-6)
        colors_float = plt.cm.jet(norm_depth)[:, :3]
        color_img = (colors_float * 255).astype(np.uint8)
    half = point_size // 2
    for ui, vi, col in zip(u, v, color_img):
        for dy in range(-half, half+1):
            for dx in range(-half, half+1):
                x = ui + dx
                y = vi + dy
                if 0 <= x < img_size[0] and 0 <= y < img_size[1]:
                    img[y, x] = col
    return img

def render_matplotlib(points, colors, elev, azim, img_size, point_size):
    """使用 matplotlib 3D scatter，适用于密集点云（带纹理颜色）"""
    if len(points) == 0:
        return np.ones((img_size[1], img_size[0], 3), dtype=np.uint8) * 240
    fig = plt.figure(figsize=(img_size[0]/100, img_size[1]/100), dpi=100, frameon=False)
    ax = fig.add_axes([0, 0, 1, 1], projection='3d')
    ax.set_facecolor('white')
    ax.set_axis_off()
    ax.view_init(elev=elev, azim=azim)
    ax.set_xlim(-1.2 * CAM_ZOOM, 1.2 * CAM_ZOOM)
    ax.set_ylim(-1.2 * CAM_ZOOM, 1.2 * CAM_ZOOM)
    ax.set_zlim(-1.2 * CAM_ZOOM, 1.2 * CAM_ZOOM)
    ax.set_box_aspect([1,1,1])
    if colors is not None and len(colors) == len(points):
        if colors.dtype == np.uint8:
            colors_float = colors / 255.0
        else:
            colors_float = colors
        ax.scatter(points[:,0], points[:,1], points[:,2],
                   c=colors_float, s=point_size, marker='.', alpha=0.8, rasterized=True)
    else:
        ax.scatter(points[:,0], points[:,1], points[:,2],
                   c='gray', s=point_size, marker='.', alpha=0.8, rasterized=True)
    fig.canvas.draw()
    if hasattr(fig.canvas, 'buffer_rgba'):
        buf = fig.canvas.buffer_rgba()
        img = np.asarray(buf)
    else:
        img = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
        img = img.reshape(fig.canvas.get_width_height()[1], fig.canvas.get_width_height()[0], 3)
    plt.close(fig)
    img_pil = Image.fromarray(img).resize((img_size[0], img_size[1]), Image.Resampling.LANCZOS)
    return np.array(img_pil)

def render_auto(points, colors, img_size, elev, azim, dist):
    """根据点数自动选择渲染方式"""
    num = len(points)
    if num < SPARSE_THRESHOLD:
        print(f"使用手动渲染 (稀疏点云, {num} 点)")
        return render_manual(points, colors, img_size, elev, azim, dist, POINT_SIZE_SPARSE)
    else:
        print(f"使用 matplotlib 渲染 (密集点云, {num} 点)")
        return render_matplotlib(points, colors, elev, azim, img_size, POINT_SIZE_DENSE)

def compute_chamfer_distance(pc1, pc2, n_sample=5000):
    """计算双向 Chamfer Distance"""
    if len(pc1) == 0 or len(pc2) == 0:
        return float('inf')
    if len(pc1) > n_sample:
        idx = np.random.choice(len(pc1), n_sample, replace=False)
        pc1 = pc1[idx]
    if len(pc2) > n_sample:
        idx = np.random.choice(len(pc2), n_sample, replace=False)
        pc2 = pc2[idx]
    tree1 = KDTree(pc1)
    tree2 = KDTree(pc2)
    dist1, _ = tree1.query(pc2)
    dist2, _ = tree2.query(pc1)
    return np.mean(dist1) + np.mean(dist2)

# ========== 收集文件并计算 CD ==========
preds = {}
gts = {}
for p in PRED_DIR.glob("*.glb"):
    c = parse_coords(p.name)
    if c: preds[c] = p
for p in GT_DIR.glob("*.glb"):
    c = parse_coords(p.name)
    if c: gts[c] = p

common = sorted(set(preds.keys()) & set(gts.keys()))
print(f"找到 {len(common)} 对共同坐标")

results = []
for (x,y) in common:
    try:
        pts_pred, col_pred = load_points_and_colors(preds[(x,y)])
        pts_gt, col_gt = load_points_and_colors(gts[(x,y)])
        pts_pred_norm = normalize_points(pts_pred)
        pts_gt_norm = normalize_points(pts_gt)
        cd = compute_chamfer_distance(pts_gt_norm, pts_pred_norm, CD_DOWNSAMPLE)
        results.append({
            'coords': (x,y),
            'cd': cd,
            'pts_pred': pts_pred_norm,
            'pts_gt': pts_gt_norm,
            'col_pred': col_pred,
            'col_gt': col_gt
        })
        print(f"坐标 ({x},{y})  Pred: {len(pts_pred)}  GT: {len(pts_gt)}  CD = {cd:.4f}")
    except Exception as e:
        print(f"处理 ({x},{y}) 出错: {e}")

results.sort(key=lambda r: r['cd'])
print(f"\n成功处理 {len(results)} 对")

# ========== Figure 6: Overview (median quality) ==========
median_start = len(results) // 2 - NUM_OVERVIEW // 2
overview_samples = results[median_start:median_start + NUM_OVERVIEW]

print("\n===== Figure 6: Overview samples =====")
for i, s in enumerate(overview_samples):
    print(f"  {i+1}: {s['coords']}  CD={s['cd']:.4f}")

fig6, axes6 = plt.subplots(4, 8, figsize=(30, 18))
for row in range(4):
    for col in range(4):
        sample = overview_samples[row * 4 + col]
        ax_pred = axes6[row, col*2]
        ax_gt = axes6[row, col*2 + 1]

        img_gt = render_auto(sample['pts_gt'], sample['col_gt'], IMG_SIZE, CAM_ELEV, CAM_AZIM, CAM_DIST)
        ax_gt.imshow(img_gt)
        ax_gt.axis('off')
        ax_gt.text(0.05, 0.95, 'GT', transform=ax_gt.transAxes,
                   fontsize=8, color='white', ha='left', va='top',
                   bbox=dict(facecolor='black', alpha=0.5, pad=1))

        img_pred = render_auto(sample['pts_pred'], sample['col_pred'], IMG_SIZE, CAM_ELEV, CAM_AZIM, CAM_DIST)
        ax_pred.imshow(img_pred)
        ax_pred.axis('off')
        ax_pred.text(0.05, 0.95, 'Predicted', transform=ax_pred.transAxes,
                     fontsize=8, color='white', ha='left', va='top',
                     bbox=dict(facecolor='black', alpha=0.5, pad=1))



plt.tight_layout()
fig6.savefig(OUT_DIR / 'pc_tile_overview.png', dpi=200, bbox_inches='tight')
plt.close()
print("Figure 6 saved.")

# # ========== Figure 7: Challenging cases (worst CD) ==========
# challenge_samples = results[-NUM_CHALLENGE:]

# print("\n===== Figure 7: Challenge samples =====")
# for i, s in enumerate(challenge_samples):
#     print(f"  {i+1}: {s['coords']}  CD={s['cd']:.4f}")

# # 布局：2 行，每行 3 对（共 6 列子图，每对占 2 列）
# n_rows = 2
# n_pairs_per_row = 3
# fig7, axes7 = plt.subplots(n_rows, n_pairs_per_row * 2, figsize=(24, 14))

# for row in range(n_rows):
#     for col in range(n_pairs_per_row):
#         idx = row * n_pairs_per_row + col
#         if idx >= len(challenge_samples):
#             # 隐藏多余子图
#             axes7[row, col*2].axis('off')
#             axes7[row, col*2+1].axis('off')
#             continue
#         sample = challenge_samples[idx]
#         ax_gt = axes7[row, col*2]
#         ax_pred = axes7[row, col*2+1]

#         img_gt = render_auto(sample['pts_gt'], sample['col_gt'], IMG_SIZE, CAM_ELEV, CAM_AZIM, CAM_DIST)
#         ax_gt.imshow(img_gt)
#         ax_gt.axis('off')
#         ax_gt.text(0.05, 0.95, 'Ground Truth', transform=ax_gt.transAxes,
#                    fontsize=8, color='white', ha='left', va='top',
#                    bbox=dict(facecolor='black', alpha=0.5, pad=1))
#         ax_gt.set_title(f"CD: {sample['cd']:.4f}", fontsize=10, pad=5)

#         img_pred = render_auto(sample['pts_pred'], sample['col_pred'], IMG_SIZE, CAM_ELEV, CAM_AZIM, CAM_DIST)
#         ax_pred.imshow(img_pred)
#         ax_pred.axis('off')
#         ax_pred.text(0.05, 0.95, 'Predicted', transform=ax_pred.transAxes,
#                      fontsize=8, color='white', ha='left', va='top',
#                      bbox=dict(facecolor='black', alpha=0.5, pad=1))

# plt.tight_layout()
# fig7.savefig(OUT_DIR / 'pc_tile_challenge.png', dpi=200, bbox_inches='tight')
# plt.close()
# print("Figure 7 saved.")

找到 105 对共同坐标
坐标 (-1450,0)  Pred: 536704  GT: 536704  CD = 0.0791
坐标 (-1450,50)  Pred: 536704  GT: 536704  CD = 0.0662
坐标 (-1450,100)  Pred: 536705  GT: 536704  CD = 0.0890
坐标 (-1450,150)  Pred: 536704  GT: 536705  CD = 0.1113
坐标 (-1450,200)  Pred: 536704  GT: 536704  CD = 0.0709
坐标 (-1450,250)  Pred: 536704  GT: 536704  CD = 0.0995
坐标 (-1450,300)  Pred: 536704  GT: 536704  CD = 0.0831
坐标 (-1400,0)  Pred: 536705  GT: 536704  CD = 0.0434
坐标 (-1400,50)  Pred: 536704  GT: 536705  CD = 0.0572
坐标 (-1400,100)  Pred: 536704  GT: 536704  CD = 0.0464
坐标 (-1400,150)  Pred: 536704  GT: 536704  CD = 0.0533
坐标 (-1400,200)  Pred: 536704  GT: 536704  CD = 0.0629
坐标 (-1400,250)  Pred: 536705  GT: 536704  CD = 0.0560
坐标 (-1400,300)  Pred: 536705  GT: 536704  CD = 0.0542
坐标 (-1350,0)  Pred: 536704  GT: 536704  CD = 0.0509
坐标 (-1350,50)  Pred: 536704  GT: 536704  CD = 0.0527
坐标 (-1350,100)  Pred: 536704  GT: 536704  CD = 0.0627
坐标 (-1350,150)  Pred: 536704  GT: 536704  CD = 0.0673
坐标 (-1350,200)  Pred: 53

KeyboardInterrupt: 

In [4]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
import trimesh
from scipy.spatial import cKDTree
from PIL import Image

# ======================== 配置 ========================
PRED_DIR = Path("/home/viplab/liukejia/project/result2/reconstruct_pred_glb")
GT_DIR   = Path("/home/viplab/liukejia/project/result2/reconstruct_gt_glb")
OUT_DIR  = Path("/home/viplab/liukejia/project/result2/figures")
OUT_DIR.mkdir(exist_ok=True, parents=True)

MAX_POINTS = 120000          # CD 计算时的最大点数（与第一份代码一致）
CD_SAMPLE = None             # 不额外下采样，直接用加载的点

# 可视化参数（与第二份代码一致）
IMG_SIZE = (512, 512)
CAM_ELEV = 30
CAM_AZIM = -45
CAM_DIST = 0.3
CAM_ZOOM = 0.4
SPARSE_THRESHOLD = 25000
POINT_SIZE_SPARSE = 5
POINT_SIZE_DENSE = 2.5

NUM_CHALLENGE = 6            # 显示 CD 最大的 6 个案例

# ======================== 点云加载（兼顾 CD 计算与渲染） ========================
def load_points_for_cd(glb_path: str, max_points: int, seed: int):
    """仅返回原始坐标（float64），用于 CD 计算，与第一份代码一致"""
    obj = trimesh.load(glb_path, force="scene")
    if isinstance(obj, trimesh.Trimesh):
        pts = np.asarray(obj.vertices, dtype=np.float64)
    else:
        merged = obj.dump(concatenate=True)
        if isinstance(merged, (list, tuple)):
            merged = merged[0] if len(merged) > 0 else None
        if merged is None or not hasattr(merged, "vertices"):
            raise ValueError(f"无法解析顶点: {glb_path}")
        pts = np.asarray(merged.vertices, dtype=np.float64)
    if pts.shape[0] == 0:
        raise ValueError(f"空点云: {glb_path}")
    if max_points is not None and pts.shape[0] > max_points:
        rng = np.random.default_rng(seed)
        idx = rng.choice(pts.shape[0], size=max_points, replace=False)
        pts = pts[idx]
    return pts

def load_points_and_colors_for_render(glb_path: str):
    """返回 (points, colors) 用于渲染（全部点，不做下采样）"""
    obj = trimesh.load(glb_path, force="scene")
    points_list = []
    colors_list = []

    def collect(item):
        if isinstance(item, trimesh.Scene):
            for geom in item.geometry.values():
                collect(geom)
        elif isinstance(item, trimesh.PointCloud):
            if len(item.vertices) > 0:
                points_list.append(item.vertices)
                if item.colors is not None:
                    colors = item.colors
                    if colors.shape[1] == 4:
                        colors = colors[:, :3]
                    if colors.max() <= 1.0:
                        colors = (colors * 255).astype(np.uint8)
                    colors_list.append(colors)
                else:
                    colors_list.append(None)
        elif isinstance(item, trimesh.Trimesh):
            if len(item.vertices) > 0:
                points_list.append(item.vertices)
                if hasattr(item.visual, 'vertex_colors') and item.visual.vertex_colors is not None:
                    vc = item.visual.vertex_colors
                    if vc.shape[1] == 4:
                        vc = vc[:, :3]
                    if vc.max() <= 1.0:
                        vc = (vc * 255).astype(np.uint8)
                    colors_list.append(vc)
                else:
                    colors_list.append(None)

    collect(obj)
    if not points_list:
        return np.empty((0, 3)), None
    all_pts = np.vstack(points_list)
    all_colors = None
    if all(c is not None for c in colors_list):
        all_colors = np.vstack(colors_list)
        if len(all_colors) != len(all_pts):
            all_colors = None
    return all_pts, all_colors

# ======================== CD 计算（与第一份代码完全一致） ========================
def chamfer_distance_l2_squared(p_rec: np.ndarray, p_gt: np.ndarray):
    """L2 平方 Chamfer Distance，原始坐标，不归一化"""
    tree_rec = cKDTree(p_rec)
    tree_gt = cKDTree(p_gt)
    d_rec, _ = tree_gt.query(p_rec)
    d_gt, _ = tree_rec.query(p_gt)
    cd = np.mean(d_rec ** 2) + np.mean(d_gt ** 2)
    return float(cd)

# ======================== 渲染函数（与第二份代码完全相同） ========================
def normalize_points(points):
    if len(points) == 0:
        return points
    center = points.mean(axis=0)
    points_centered = points - center
    max_radius = np.linalg.norm(points_centered, axis=1).max()
    if max_radius > 0:
        points_centered /= max_radius
    return points_centered

def project_points(points, elev, azim, dist, img_size):
    if len(points) == 0:
        return np.array([]), np.array([]), np.array([])
    elev_rad = np.radians(elev)
    azim_rad = np.radians(azim)
    cx = dist * np.cos(elev_rad) * np.cos(azim_rad)
    cy = dist * np.cos(elev_rad) * np.sin(azim_rad)
    cz = dist * np.sin(elev_rad)
    forward = -np.array([cx, cy, cz])
    forward = forward / np.linalg.norm(forward)
    world_up = np.array([0, 1, 0])
    right = np.cross(world_up, forward)
    right = right / np.linalg.norm(right)
    up = np.cross(forward, right)
    R = np.vstack([right, up, forward])
    points_cam = points - np.array([cx, cy, cz])
    points_cam = points_cam @ R.T
    fov_rad = np.radians(45)
    focal = 1.0 / np.tan(fov_rad/2)
    x_proj = points_cam[:,0] * focal / points_cam[:,2]
    y_proj = points_cam[:,1] * focal / points_cam[:,2]
    width, height = img_size
    u = (x_proj + 1.0) * 0.5 * width
    v = (1.0 - (y_proj + 1.0) * 0.5) * height
    valid = (points_cam[:,2] > 0.01) & (u >= 0) & (u < width) & (v >= 0) & (v < height)
    return u[valid], v[valid], points_cam[valid, 2]

def render_manual(points, colors, img_size, elev, azim, dist, point_size):
    u, v, depth = project_points(points, elev, azim, dist, img_size)
    if len(u) == 0:
        return np.ones((img_size[1], img_size[0], 3), dtype=np.uint8) * 240
    order = np.argsort(depth)[::-1]
    u = u[order].astype(int)
    v = v[order].astype(int)
    img = np.ones((img_size[1], img_size[0], 3), dtype=np.uint8) * 255
    if colors is not None and len(colors) == len(points):
        colors_sorted = colors[order]
        if colors_sorted.dtype != np.uint8:
            if colors_sorted.max() <= 1.0:
                colors_sorted = (colors_sorted * 255).astype(np.uint8)
            else:
                colors_sorted = colors_sorted.astype(np.uint8)
        color_img = colors_sorted
    else:
        norm_depth = (depth[order] - depth.min()) / (depth.max() - depth.min() + 1e-6)
        colors_float = plt.cm.jet(norm_depth)[:, :3]
        color_img = (colors_float * 255).astype(np.uint8)
    half = point_size // 2
    for ui, vi, col in zip(u, v, color_img):
        for dy in range(-half, half+1):
            for dx in range(-half, half+1):
                x = ui + dx
                y = vi + dy
                if 0 <= x < img_size[0] and 0 <= y < img_size[1]:
                    img[y, x] = col
    return img

def render_matplotlib(points, colors, elev, azim, img_size, point_size):
    if len(points) == 0:
        return np.ones((img_size[1], img_size[0], 3), dtype=np.uint8) * 240
    fig = plt.figure(figsize=(img_size[0]/100, img_size[1]/100), dpi=100, frameon=False)
    ax = fig.add_axes([0, 0, 1, 1], projection='3d')
    ax.set_facecolor('white')
    ax.set_axis_off()
    ax.view_init(elev=elev, azim=azim)
    ax.set_xlim(-1.2 * CAM_ZOOM, 1.2 * CAM_ZOOM)
    ax.set_ylim(-1.2 * CAM_ZOOM, 1.2 * CAM_ZOOM)
    ax.set_zlim(-1.2 * CAM_ZOOM, 1.2 * CAM_ZOOM)
    ax.set_box_aspect([1,1,1])
    if colors is not None and len(colors) == len(points):
        if colors.dtype == np.uint8:
            colors_float = colors / 255.0
        else:
            colors_float = colors
        ax.scatter(points[:,0], points[:,1], points[:,2],
                   c=colors_float, s=point_size, marker='.', alpha=0.8, rasterized=True)
    else:
        ax.scatter(points[:,0], points[:,1], points[:,2],
                   c='gray', s=point_size, marker='.', alpha=0.8, rasterized=True)
    fig.canvas.draw()
    if hasattr(fig.canvas, 'buffer_rgba'):
        buf = fig.canvas.buffer_rgba()
        img = np.asarray(buf)
    else:
        img = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
        img = img.reshape(fig.canvas.get_width_height()[1], fig.canvas.get_width_height()[0], 3)
    plt.close(fig)
    img_pil = Image.fromarray(img).resize((img_size[0], img_size[1]), Image.Resampling.LANCZOS)
    return np.array(img_pil)

def render_auto(points, colors, img_size, elev, azim, dist):
    num = len(points)
    if num < SPARSE_THRESHOLD:
        print(f"使用手动渲染 (稀疏点云, {num} 点)")
        return render_manual(points, colors, img_size, elev, azim, dist, POINT_SIZE_SPARSE)
    else:
        print(f"使用 matplotlib 渲染 (密集点云, {num} 点)")
        return render_matplotlib(points, colors, elev, azim, img_size, POINT_SIZE_DENSE)

# ======================== 主流程 ========================
# 收集所有配对文件
pred_files = {p.name: p for p in PRED_DIR.glob("*.glb")}
gt_files   = {p.name: p for p in GT_DIR.glob("*.glb")}
common_names = sorted(set(pred_files.keys()) & set(gt_files.keys()))
print(f"找到 {len(common_names)} 对共同文件")

# 计算正确的 CD（使用原始坐标、L2 平方、固定下采样）
results = []
for i, name in enumerate(common_names):
    try:
        # 用于 CD 计算的点云（原始坐标，下采样至 MAX_POINTS，随机种子固定）
        pts_pred_cd = load_points_for_cd(pred_files[name], MAX_POINTS, seed=1000 + 10*i)
        pts_gt_cd   = load_points_for_cd(gt_files[name],   MAX_POINTS, seed=1000 + 10*i + 1)

        cd = chamfer_distance_l2_squared(pts_pred_cd, pts_gt_cd)

        # 用于渲染的点云（全部点，含颜色，后续会归一化）
        pts_pred_render, col_pred = load_points_and_colors_for_render(pred_files[name])
        pts_gt_render,   col_gt   = load_points_and_colors_for_render(gt_files[name])

        # 渲染前归一化
        pts_pred_render_norm = normalize_points(pts_pred_render)
        pts_gt_render_norm   = normalize_points(pts_gt_render)

        results.append({
            'name': name,
            'cd': cd,
            'pts_pred': pts_pred_render_norm,
            'pts_gt': pts_gt_render_norm,
            'col_pred': col_pred,
            'col_gt': col_gt
        })
        print(f"处理: {name}  |  Pred: {len(pts_pred_render)} 点, GT: {len(pts_gt_render)} 点, CD = {cd:.6f}")
    except Exception as e:
        print(f"处理 {name} 出错: {e}")

# 按 CD 从大到小排序，取最差的 NUM_CHALLENGE 个
results.sort(key=lambda r: r['cd'], reverse=True)

# ========== Figure 6: Overview (median quality) ==========
NUM_OVERVIEW = 16
median_start = len(results) // 2 - NUM_OVERVIEW // 2
overview_samples = results[median_start:median_start + NUM_OVERVIEW]

print("\n===== Figure 6: Overview samples =====")
for i, s in enumerate(overview_samples):
    print(f"  {i+1}: {s['name']}  CD={s['cd']:.4f}")

fig6, axes6 = plt.subplots(4, 8, figsize=(30, 18))
for row in range(4):
    for col in range(4):
        sample = overview_samples[row * 4 + col]
        ax_pred = axes6[row, col*2]
        ax_gt = axes6[row, col*2 + 1]

        img_gt = render_auto(sample['pts_gt'], sample['col_gt'], IMG_SIZE, CAM_ELEV, CAM_AZIM, CAM_DIST)
        ax_gt.imshow(img_gt)
        ax_gt.axis('off')
        ax_gt.text(0.05, 0.95, 'GT', transform=ax_gt.transAxes,
                   fontsize=8, color='white', ha='left', va='top',
                   bbox=dict(facecolor='black', alpha=0.5, pad=1))

        img_pred = render_auto(sample['pts_pred'], sample['col_pred'], IMG_SIZE, CAM_ELEV, CAM_AZIM, CAM_DIST)
        ax_pred.imshow(img_pred)
        ax_pred.axis('off')
        ax_pred.text(0.05, 0.95, 'Predicted', transform=ax_pred.transAxes,
                     fontsize=8, color='white', ha='left', va='top',
                     bbox=dict(facecolor='black', alpha=0.5, pad=1))



plt.tight_layout()
fig6.savefig(OUT_DIR / 'pc_tile_overview.png', dpi=200, bbox_inches='tight')
plt.close()
print("Figure 6 saved.")

# ========== Figure 7: Challenge Cases (worst CD) ==========
challenge_samples = results[:NUM_CHALLENGE]

print("\n===== CD 最差的 {} 个案例 (与第一份代码一致) =====".format(NUM_CHALLENGE))
for i, s in enumerate(challenge_samples):
    print(f"  {i+1}: {s['name']}  CD={s['cd']:.6f}")

if len(challenge_samples) == 0:
    print("没有可用样本，退出。")
    exit()

# 生成图 7（布局 2 行 x 3 列，每列包含 GT 和 Pred 两幅子图）
n_rows = 2
n_pairs_per_row = 3
fig7, axes7 = plt.subplots(n_rows, n_pairs_per_row * 2, figsize=(24, 14))

for row in range(n_rows):
    for col in range(n_pairs_per_row):
        idx = row * n_pairs_per_row + col
        if idx >= len(challenge_samples):
            axes7[row, col*2].axis('off')
            axes7[row, col*2+1].axis('off')
            continue

        sample = challenge_samples[idx]
        ax_gt   = axes7[row, col*2]
        ax_pred = axes7[row, col*2+1]

        # GT 渲染
        img_gt = render_auto(sample['pts_gt'], sample['col_gt'],
                             IMG_SIZE, CAM_ELEV, CAM_AZIM, CAM_DIST)
        ax_gt.imshow(img_gt)
        ax_gt.axis('off')
        ax_gt.text(0.05, 0.95, 'GT', transform=ax_gt.transAxes,
                   fontsize=9, color='white', ha='left', va='top',
                   bbox=dict(facecolor='black', alpha=0.5, pad=2))
        ax_gt.set_title(f"CD: {sample['cd']:.4f}", fontsize=10, pad=5)

        # Pred 渲染
        img_pred = render_auto(sample['pts_pred'], sample['col_pred'],
                               IMG_SIZE, CAM_ELEV, CAM_AZIM, CAM_DIST)
        ax_pred.imshow(img_pred)
        ax_pred.axis('off')
        ax_pred.text(0.05, 0.95, 'Predicted', transform=ax_pred.transAxes,
                     fontsize=9, color='white', ha='left', va='top',
                     bbox=dict(facecolor='black', alpha=0.5, pad=2))

plt.tight_layout()
output_path = OUT_DIR / 'pc_tile_challenge.png'
fig7.savefig(output_path, dpi=200, bbox_inches='tight')
plt.close()
print(f"\nFigure 7 已保存至: {output_path}")

找到 105 对共同文件
处理: num123_-1000_0_0_0064.glb  |  Pred: 536704 点, GT: 536704 点, CD = 0.023007
处理: num123_-1000_100_0_0066.glb  |  Pred: 536704 点, GT: 536704 点, CD = 1.962086


/tmp/ipykernel_2666318/3735438017.py:38: DeprecationWarning: `Scene.dump(concatenate=True)` DEPRECATED FOR REMOVAL APRIL 2025: replace with `Scene.to_geometry()`
  merged = obj.dump(concatenate=True)


处理: num123_-1000_150_0_0067.glb  |  Pred: 536704 点, GT: 536704 点, CD = 0.669844
处理: num123_-1000_200_0_0068.glb  |  Pred: 536704 点, GT: 536705 点, CD = 0.501413
处理: num123_-1000_250_0_0069.glb  |  Pred: 536708 点, GT: 536704 点, CD = 200.301988
处理: num123_-1000_300_0_0070.glb  |  Pred: 536704 点, GT: 536704 点, CD = 0.360941
处理: num123_-1000_50_0_0065.glb  |  Pred: 536704 点, GT: 536704 点, CD = 0.023159
处理: num123_-1050_0_0_0057.glb  |  Pred: 536704 点, GT: 536704 点, CD = 0.560701
处理: num123_-1050_100_0_0059.glb  |  Pred: 536704 点, GT: 536704 点, CD = 2.094979
处理: num123_-1050_150_0_0060.glb  |  Pred: 536704 点, GT: 536704 点, CD = 1.487971
处理: num123_-1050_200_0_0061.glb  |  Pred: 536704 点, GT: 536705 点, CD = 0.309340
处理: num123_-1050_250_0_0062.glb  |  Pred: 536704 点, GT: 536704 点, CD = 4.391903
处理: num123_-1050_300_0_0063.glb  |  Pred: 536705 点, GT: 536704 点, CD = 0.402309
处理: num123_-1050_50_0_0058.glb  |  Pred: 536704 点, GT: 536704 点, CD = 0.112265
处理: num123_-1100_0_0_0050.glb  |  Pred: 53

In [10]:
# %% [markdown]
# # 点云评估：CD ↓, RMSE ↓, Recall@2m ↑
# 
# **方法**：hsi, no_deformable, rgb  
# **GT**：hsi_grid5_32500/reconstruct_gt_glb

# %%
import numpy as np
import pandas as pd
from pathlib import Path
import open3d as o3d
from scipy.spatial import KDTree
import re
import os

# ================================================================
# 1. 配置（请根据实际情况修改）
# ================================================================
FOLDER_METHOD1 = "/data/results/result5/hsi_grid5_32500/reconstruct_pred_glb"   # hsi
FOLDER_METHOD2 = "/data/results/result5/hsi_no_deformable_grid5_15000/reconstruct_pred_glb"   # no_deformable
FOLDER_METHOD3 = "/data/results/result5/rgb_grid5_35000/reconstruct_pred_glb"   # rgb
FOLDER_GT      = "/data/results/result5/hsi_grid5_32500/reconstruct_gt_glb"    # Ground Truth

METHOD_NAMES = ["hsi", "no_deformable", "rgb"]   # 显示名称

# 手动指定场景编号（留空则自动从 GT 文件夹中提取）
SCENE_NUMBERS = []   # 例如 [84, 198, 215, 221, 401]

# 点云采样点数（当读取 GLB/GLTF 网格时）
NUM_SAMPLES = 120000

# Recall 阈值（单位：米）
THRESHOLD = 0.2

# ================================================================
# 2. 诊断：查看 GT 文件夹中的文件
# ================================================================
gt_path = Path(FOLDER_GT)
print(f"GT 文件夹: {gt_path}")
print("文件列表（并尝试提取场景编号）:")
files_found = []
for f in gt_path.glob('*.*'):
    files_found.append(f.name)
    # 尝试多种匹配模式
    num = None
    for pattern in [r'num_(\d+)', r'num(\d+)', r'scene_(\d+)', r'scene(\d+)', r'(\d+)']:
        match = re.search(pattern, f.name)
        if match:
            num = int(match.group(1))
            break
    print(f"  {f.name}  -> num={num}")
print("="*60)

# ================================================================
# 3. 提取场景编号（自动或手动）
# ================================================================
def extract_scene_number(filename):
    for pattern in [r'num_(\d+)', r'num(\d+)', r'scene_(\d+)', r'scene(\d+)']:
        match = re.search(pattern, filename)
        if match:
            return int(match.group(1))
    return None

if not SCENE_NUMBERS:
    scene_numbers = []
    for f in gt_path.glob('*.*'):
        num = extract_scene_number(f.name)
        if num is not None:
            scene_numbers.append(num)
    if not scene_numbers:
        print("⚠️ 未从 GT 文件夹提取到任何场景编号。")
        print("请检查文件名格式，或手动设置 SCENE_NUMBERS 列表（例如 [1,2,3,...]）。")
        print("可用的文件:", files_found)
        raise ValueError("无法自动提取场景编号，请手动指定 SCENE_NUMBERS")
else:
    scene_numbers = SCENE_NUMBERS

scene_numbers = sorted(set(scene_numbers))
print(f"共 {len(scene_numbers)} 个场景: {scene_numbers[:20]}{'...' if len(scene_numbers)>20 else ''}")

# ================================================================
# 4. 加载点云 / 网格函数
# ================================================================
def load_pointcloud(filepath, num_samples=10000):
    ext = filepath.suffix.lower()
    if ext in ('.ply', '.pcd'):
        pcd = o3d.io.read_point_cloud(str(filepath))
        return np.asarray(pcd.points)
    elif ext == '.npy':
        return np.load(filepath)
    elif ext in ('.txt', '.xyz'):
        return np.loadtxt(filepath)
    elif ext in ('.glb', '.gltf'):
        mesh = o3d.io.read_triangle_mesh(str(filepath))
        if not mesh.has_vertices():
            raise ValueError("网格无顶点")
        pcd = mesh.sample_points_uniformly(number_of_points=num_samples)
        return np.asarray(pcd.points)
    else:
        raise ValueError(f"不支持的文件格式: {ext}")

def compute_metrics(pred_pts, gt_pts, threshold=2.0):
    if len(pred_pts) == 0 or len(gt_pts) == 0:
        return np.nan, np.nan, np.nan
    tree_gt = KDTree(gt_pts)
    dist_pred_to_gt, _ = tree_gt.query(pred_pts)
    mean_dist_pred = np.mean(dist_pred_to_gt)
    rmse = np.sqrt(np.mean(dist_pred_to_gt ** 2))
    recall = np.mean(dist_pred_to_gt < threshold)
    tree_pred = KDTree(pred_pts)
    dist_gt_to_pred, _ = tree_pred.query(gt_pts)
    mean_dist_gt = np.mean(dist_gt_to_pred)
    cd = (mean_dist_pred + mean_dist_gt) / 2.0
    return cd, rmse, recall

# ================================================================
# 5. 主评估循环
# ================================================================
method_folders = [Path(FOLDER_METHOD1), Path(FOLDER_METHOD2), Path(FOLDER_METHOD3)]
gt_folder = Path(FOLDER_GT)

results = []
for scene in scene_numbers:
    # 查找 GT 文件
    gt_file = None
    for f in gt_folder.glob(f'*{scene:04d}*.*'):
        gt_file = f
        break
    if gt_file is None:
        # 尝试不加零填充
        for f in gt_folder.glob(f'*{scene}*.*'):
            gt_file = f
            break
    if gt_file is None:
        print(f"场景 {scene} 的 GT 文件未找到，跳过")
        continue

    try:
        gt_pts = load_pointcloud(gt_file, NUM_SAMPLES)
        print(f"场景 {scene}: GT 点数 {gt_pts.shape[0]}")
    except Exception as e:
        print(f"读取 GT {scene} 失败: {e}")
        continue

    row = {'scene': scene}
    for idx, mf in enumerate(method_folders):
        method_name = METHOD_NAMES[idx]
        method_file = None
        # 匹配文件名时同样尝试多种模式
        for f in mf.glob(f'*{scene:04d}*.*'):
            method_file = f
            break
        if method_file is None:
            for f in mf.glob(f'*{scene}*.*'):
                method_file = f
                break
        if method_file is None:
            print(f"场景 {scene} 方法 {method_name} 文件未找到，跳过")
            continue

        try:
            pred_pts = load_pointcloud(method_file, NUM_SAMPLES)
            cd, rmse, recall = compute_metrics(pred_pts, gt_pts, THRESHOLD)
            row[f'{method_name}_cd'] = cd
            row[f'{method_name}_rmse'] = rmse
            row[f'{method_name}_recall'] = recall
            print(f"  {method_name}: CD={cd:.4f}, RMSE={rmse:.4f}, Recall={recall:.4f}")
        except Exception as e:
            print(f"处理 {method_file} 失败: {e}")
            continue

    results.append(row)

# ================================================================
# 6. 结果展示
# ================================================================
if results:
    df = pd.DataFrame(results)
    print("\n=== 详细结果（所有场景）===")
    # 显示前10行，若场景很多则显示前10行和后10行
    if len(df) > 20:
        display(df.head(10))
        print("...")
        display(df.tail(10))
    else:
        display(df)

    print("\n=== 汇总统计（均值 ± 标准差）===")
    summary = {}
    for name in METHOD_NAMES:
        metrics = {}
        for metric in ['cd', 'rmse', 'recall']:
            col = f'{name}_{metric}'
            if col in df.columns:
                mean_val = df[col].mean()
                std_val = df[col].std()
                metrics[metric] = f"{mean_val:.6f} ± {std_val:.6f}"
        summary[name] = metrics
    summary_df = pd.DataFrame(summary).T
    display(summary_df)

    # 保存 CSV
    df.to_csv('metrics.csv', index=False)
    print("\n完整结果已保存至 metrics.csv")
else:
    print("没有成功评估任何场景，请检查文件路径和命名。")

GT 文件夹: /data/results/result5/hsi_grid5_32500/reconstruct_gt_glb
文件列表（并尝试提取场景编号）:
  num123_-800_250_0_0097.glb  -> num=123
  num123_-950_200_0_0075.glb  -> num=123
  num123_-1300_250_0_0027.glb  -> num=123
  num123_-1250_150_0_0032.glb  -> num=123
  num123_-1350_300_0_0021.glb  -> num=123
  num123_-1150_100_0_0045.glb  -> num=123
  num123_-1300_150_0_0025.glb  -> num=123
  num123_-1050_250_0_0062.glb  -> num=123
  num123_-1250_0_0_0029.glb  -> num=123
  num123_-1100_0_0_0050.glb  -> num=123
  num123_-1200_150_0_0039.glb  -> num=123
  num123_-900_200_0_0082.glb  -> num=123
  num123_-1150_300_0_0049.glb  -> num=123
  num123_-1100_250_0_0055.glb  -> num=123
  num123_-750_0_0_0099.glb  -> num=123
  num123_-1150_0_0_0043.glb  -> num=123
  num123_-1450_0_0_0001.glb  -> num=123
  num123_-1350_200_0_0019.glb  -> num=123
  num123_-1200_200_0_0040.glb  -> num=123
  num123_-1400_100_0_0010.glb  -> num=123
  num123_-1050_300_0_0063.glb  -> num=123
  num123_-1400_200_0_0012.glb  -> num=123
  num123

,scene,hsi_cd,hsi_rmse,hsi_recall,no_deformable_cd,no_deformable_rmse,no_deformable_recall,rgb_cd,rgb_rmse,rgb_recall
0,123,0.195799,0.376205,0.743108,0.10183,0.175743,0.757225,0.211458,0.409853,0.73855



=== 汇总统计（均值 ± 标准差）===


,cd,rmse,recall
hsi,0.195799 ± nan,0.376205 ± nan,0.743108 ± nan
no_deformable,0.101830 ± nan,0.175743 ± nan,0.757225 ± nan
rgb,0.211458 ± nan,0.409853 ± nan,0.738550 ± nan



完整结果已保存至 metrics.csv
